# IC 4040 – SIMBAD Downloader

<div class="alert alert-block alert-info">
<b>Environment:</b> Run this notebook in the <code>stenv</code> conda environment.
</div>

## Imports

In [ ]:
# Python Imports
import os
from pathlib import Path

# Astropy Collaboration Imports
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.table import QTable, Table
from astroquery.simbad import Simbad
from regions import PointSkyRegion, Regions

## Notebook Setup

In [ ]:
if Path.cwd().name != "SIMBAD":
    if Path.cwd().name == "Notebooks":
        os.chdir("../Data/SIMBAD")
    else:
        raise RuntimeError(
            "This notebook must be run from the SIMBAD directory."
        )
print(f'Current Directory: {Path.cwd()}')

In [ ]:
# Input data
NED_DATA_FILE = Path('../NED/IC4040-NED_Data.ecsv')

# Query parameters
FOV_RADIUS_ARCMIN = 20.0  # arcminutes — adjust to match image FOV

# Output file paths
OUT_CATALOG_ECSV = Path('IC4040-SIMBAD-AlignmentStars.ecsv')
OUT_COORDS_ECSV = Path('IC4040-SIMBAD-AlignmentStars-Coordinates.ecsv')
OUT_COORDS_REG = Path('IC4040-SIMBAD-AlignmentStars-Coordinates.reg')
OUT_TWEAKREG_CAT = Path('IC4040-SIMBAD-RefCatalog-icrs.txt')

## Load External Data

Load the NED galaxy data table to obtain the galaxy's sky coordinates,
which define the centre of the SIMBAD search cone.

In [ ]:
ned_data_table = Table.read(NED_DATA_FILE)
gal_crd = SkyCoord(
    ra=ned_data_table['RA'][0],
    dec=ned_data_table['DEC'][0],
    unit='deg',
    frame='fk5'
)
print(f'Galaxy Coordinates (FK5): {gal_crd}')
print(f'Galaxy Coordinates (ICRS): {gal_crd.icrs}')

## Query SIMBAD

Query the SIMBAD database for all objects within `FOV_RADIUS_ARCMIN` arcminutes
of the galaxy centre.  The `otype` votable field is requested so that results
can be filtered to stellar point sources suitable for TweakReg alignment.

In [ ]:
# Configure SIMBAD to return the object type
simbad = Simbad()
simbad.add_votable_fields('otype')

# Run the cone search
obj_table = simbad.query_region(
    gal_crd.icrs,
    radius=FOV_RADIUS_ARCMIN * u.arcmin
)

print(f'Found {len(obj_table):d} total objects in the FOV')

In [ ]:
# Print object-type breakdown before filtering
print('Object type breakdown (all sources):')
for otype in sorted(set(str(t) for t in obj_table['otype'])):
    n = sum(1 for t in obj_table['otype'] if str(t) == otype)
    print(f'  {otype:<20s} {n:>4d}')

In [ ]:
# Build SkyCoord array from the filtered table.
# SIMBAD returns RA as a sexagesimal string ("HH MM SS.sss") and
# DEC as a sexagesimal string ("DD MM SS.ss") by default.
obj_coords = SkyCoord(
    ra=obj_table['ra'],
    dec=obj_table['dec'],
    unit=(u.hourangle, u.deg),
    frame='icrs'
)

print(f'Coordinate array built: {len(obj_coords):d} entries')

## Write Outputs

Write the filtered stellar catalogue to four output files:

1. **ECSV catalogue** — full `obj_table` with SIMBAD metadata.
2. **Coordinates ECSV** — `SkyCoord` column only, for convenient re-loading.
3. **DS9 region file** — point markers for visual inspection.
4. **TweakReg plain-text catalogue** — whitespace-separated RA Dec (ICRS decimal
   degrees) as required by `TweakReg`.

In [ ]:
# 1. Write the full filtered table
obj_table.write(OUT_CATALOG_ECSV, overwrite=True)
print(f'Wrote full catalogue  -> {OUT_CATALOG_ECSV}')

In [ ]:
# 2. Write SkyCoord-only ECSV
# Reload with: QTable.read(OUT_COORDS_ECSV)['SkyCoord']
QTable([obj_coords], names=['SkyCoord']).write(
    OUT_COORDS_ECSV, overwrite=True
)
print(f'Wrote coordinates     -> {OUT_COORDS_ECSV}')

In [ ]:
# 3. Write DS9 region file with cyan 'x' markers
regs = Regions([])
for crd in obj_coords:
    regs.append(PointSkyRegion(crd))
regs.write(str(OUT_COORDS_REG), overwrite=True)

# Insert global style directive after the first line
with open(OUT_COORDS_REG, 'r') as fid:
    lines = fid.readlines()
lines.insert(1, 'global point=x color=cyan\n')
with open(OUT_COORDS_REG, 'w') as fid:
    fid.writelines(lines)

print(f'Wrote DS9 regions     -> {OUT_COORDS_REG}')

In [ ]:
# 4. Write TweakReg plain-text catalogue (whitespace-separated RA Dec)
# TweakReg requires: 'text files containing whitespace-separated list of values'
with open(OUT_TWEAKREG_CAT, 'w') as fid:
    for crd in obj_coords:
        fid.write(
            f'{crd.icrs.ra.value:<20.8f} {crd.icrs.dec.value:<20.8f}\n'
        )

print(f'Wrote TweakReg cat.   -> {OUT_TWEAKREG_CAT}')
print(f'Total alignment stars: {len(obj_coords):d}')

## Combined GAIA + SIMBAD Catalog

Cross-match the SIMBAD stellar catalogue against the pre-existing GAIA catalogue.
Sources present in both catalogues retain their GAIA positions (higher astrometric
precision); SIMBAD-only sources are appended to the GAIA list.  The same four
output formats are written as for the individual catalogues.

In [ ]:
# GAIA input (full table required for source designations)
GAIA_TABLE_FILE = Path('../GAIA/IC4040-GAIA-AlignmentStars.ecsv')

# Combined output file paths
OUT_COMB_DIR = Path('Combined')
OUT_COMB_CATALOG_ECSV = OUT_COMB_DIR / 'IC4040-Combined-AlignmentStars.ecsv'
OUT_COMB_COORDS_ECSV = OUT_COMB_DIR / 'IC4040-Combined-AlignmentStars-Coordinates.ecsv'
OUT_COMB_COORDS_REG = OUT_COMB_DIR / 'IC4040-Combined-AlignmentStars-Coordinates.reg'
OUT_COMB_TWEAKREG_CAT = OUT_COMB_DIR / 'IC4040-Combined-RefCatalog-icrs.txt'

In [ ]:
# Make the Combined Dir
OUT_COMB_DIR.mkdir(exist_ok=True)

### Cross-Match SIMBAD Against GAIA by Name

Load the full GAIA catalogue to obtain each source's `designation`
(e.g. `Gaia DR3 1234567890123456789`).  Pass those designations to
`Simbad.query_objects()` in a single bulk call; SIMBAD returns the
canonical `main_id` for every designation it recognises.  Any SIMBAD
object in `obj_table` whose `main_id` appears in that result set is
already represented by a GAIA entry; all remaining SIMBAD objects
(stars, galaxies, nebulae, etc.) are SIMBAD-only sources and are
appended to the GAIA list.

In [ ]:
# Load full GAIA table for designations and coordinates
gaia_full_table = Table.read(GAIA_TABLE_FILE)
gaia_coords = SkyCoord(
    ra=gaia_full_table['ra'],
    dec=gaia_full_table['dec'],
    unit='deg',
    frame='icrs'
)
gaia_designations = [str(d) for d in gaia_full_table['designation']]
print(f'Loaded {len(gaia_coords):d} GAIA sources')

# Query SIMBAD in bulk by GAIA designation.
# Returns a table of SIMBAD entries (main_id) for recognised designations;
# unrecognised designations are simply absent from the result.
print(f'Querying SIMBAD for {len(gaia_designations):d} GAIA designations ...')
simbad_gaia_result = Simbad().query_objects(gaia_designations)

# Build set of SIMBAD main_ids that are known to have a GAIA counterpart
gaia_simbad_main_ids: set[str] = set()
if simbad_gaia_result is not None:
    for mid in simbad_gaia_result['main_id']:
        gaia_simbad_main_ids.add(str(mid))
print(f'GAIA sources found in SIMBAD : {len(gaia_simbad_main_ids):d}')

# SIMBAD-only objects: every obj_table entry (all types) whose main_id was
# NOT returned by the GAIA lookup
simbad_only_bool = [
    str(mid) not in gaia_simbad_main_ids
    for mid in obj_table['main_id']
]
simbad_only_coords = obj_coords[simbad_only_bool]
n_simbad_only = sum(simbad_only_bool)

print(f'SIMBAD-only sources          : {n_simbad_only:d}')
print(f'Total combined sources       : {len(gaia_coords) + n_simbad_only:d}')

# Build combined SkyCoord: all GAIA positions first, then SIMBAD-only
combined_coords = SkyCoord(
    ra=[*gaia_coords.ra.deg, *simbad_only_coords.ra.deg],
    dec=[*gaia_coords.dec.deg, *simbad_only_coords.dec.deg],
    unit='deg',
    frame='icrs'
)
source_labels = ['GAIA'] * len(gaia_coords) + ['SIMBAD'] * n_simbad_only

### Write Combined Outputs

Write the combined catalogue to the same four formats as the individual
SIMBAD and GAIA catalogues.  The full ECSV catalogue includes a `source`
column indicating whether each position originated from GAIA or SIMBAD.

In [ ]:
# 1. Write the combined catalogue with ra, dec, and source flag
comb_catalog = QTable(
    [combined_coords.ra, combined_coords.dec, source_labels],
    names=['ra', 'dec', 'source']
)
comb_catalog.write(OUT_COMB_CATALOG_ECSV, overwrite=True)
print(f'Wrote combined catalogue  -> {OUT_COMB_CATALOG_ECSV}')

In [ ]:
# 2. Write SkyCoord-only ECSV
# Reload with: QTable.read(OUT_COMB_COORDS_ECSV)['SkyCoord']
QTable([combined_coords], names=['SkyCoord']).write(
    OUT_COMB_COORDS_ECSV, overwrite=True
)
print(f'Wrote coordinates         -> {OUT_COMB_COORDS_ECSV}')

In [ ]:
# 3. Write DS9 region file with cyan 'x' markers
comb_regs = Regions([])
for crd in combined_coords:
    comb_regs.append(PointSkyRegion(crd))
comb_regs.write(str(OUT_COMB_COORDS_REG), overwrite=True)

# Insert global style directive after the first line
with open(OUT_COMB_COORDS_REG, 'r') as fid:
    lines = fid.readlines()
lines.insert(1, 'global point=x color=cyan\n')
with open(OUT_COMB_COORDS_REG, 'w') as fid:
    fid.writelines(lines)

print(f'Wrote DS9 regions         -> {OUT_COMB_COORDS_REG}')

In [ ]:
# 4. Write TweakReg plain-text catalogue (whitespace-separated RA Dec)
with open(OUT_COMB_TWEAKREG_CAT, 'w') as fid:
    for crd in combined_coords:
        fid.write(
            f'{crd.icrs.ra.value:<20.8f} {crd.icrs.dec.value:<20.8f}\n'
        )

print(f'Wrote TweakReg cat.       -> {OUT_COMB_TWEAKREG_CAT}')
print(f'Total combined stars:      {len(combined_coords):d}')